# Tokenizers for LLMs

## 1. Byte Pair Encoding (BPE)


Byte Pair Encoding (BPE) is the "secret sauce" that helps Large Language Models (LLMs) handle the massive diversity of human language without needing a dictionary the size of a planet.

Think of it as a middle ground: it’s more efficient than character-level encoding (which is too granular) and more flexible than word-level encoding (which breaks when it sees a typo or a new word).


**The Core Logic**

BPE starts with individual characters and iteratively merges the most frequently occurring adjacent pair of tokens into a single new token.

The Step-by-Step Algorithm:
1. Tokenize the text into individual characters (e.g., UTF-8 bytes).
2. Count all adjacent pairs of tokens.
3. Identify the most frequent pair (e.g., 't' and 'h' appearing together most often).
4. Merge that pair into a new token ('th').
5. Repeat until you reach a desired vocabulary size.


In modern LLMs, we don't use raw Unicode characters because there are over 140,000 of them. Instead, we convert text into `UTF-8` bytes. This ensures the base vocabulary is always exactly 256, regardless of the language (English, Chinese, or Emojis).
* Pro Tip: If you've ever seen an LLM struggle with a simple math problem like "How many 'r's are in 'strawberry'?", it's usually because the BPE tokenizer turned "strawberry" into tokens like [straw, berry]. The model "sees" the tokens, not the individual letters!


`UTF-8` encodes code points into `1-4 bytes`, depending on the balue of the code point.
* Other encodings like `utf-16` are possible, but fixed length encodings lead to wasted space.


In [1]:
msg = "Hello World. My name is Jae."

# raw bytes that represeent this string in utf-8 encoding
list(msg.encode("utf-8"))

[72,
 101,
 108,
 108,
 111,
 32,
 87,
 111,
 114,
 108,
 100,
 46,
 32,
 77,
 121,
 32,
 110,
 97,
 109,
 101,
 32,
 105,
 115,
 32,
 74,
 97,
 101,
 46]

This results in a pretty large sequence of raw decimal representations of the utf-8 byte encoding.

BUT this poses a problem because we have a limited context window (sequence length) for our model.

**QUESTION**: So, how can we create a numerical representation that is not as verbose and long, but still capture the meat of the text?

**ANSWER**: Compression

## Byte Pair Encoding
`byte-pair encoding` (BPE),or digram coding, is an algorithm, first described in 1994 by Philip Gage, for encoding strings of text into smaller strings by creating and using a translation table.[4] A slightly modified version of the algorithm is used in large language model tokenizers.

The original version of the algorithm focused on compression. It replaces the highest-frequency pair of bytes with a new byte that was not contained in the initial dataset. A lookup table of the replacements is required to rebuild the initial dataset. 

The modified version builds "tokens" (units of recognition) that match varying amounts of source text, from single characters (including single digits or single punctuation marks) to whole words (even long compound words).


Traditional Example
-------

"aaabdaaabac"

The byte pair "aa" occurs most often, so it will be replaced by a byte that is not used in the data, such as "Z". Now there is the following data and replacement table: 

"ZabdZabac"

"Z"="aa"

Then the process is repeated with byte pair "ab", replacing it with "Y": 

"ZYdZYac"

"Y"="ab"

"Z"="aa"

The only literal byte pair left occurs only once, and the encoding might stop here. 
Alternatively, the process could continue with recursive byte-pair encoding, replacing "ZY" with "X": 

"XdXac"

"X"="ZY"

"Y"="ab"

"Z"="aa"


Modified Modern Example
-------
The original BPE algorithm is modified for use in language modeling, especially for large language models based on neural networks. Compared to the original BPE, the modified BPE does not aim to maximally compress text, but rather, to encode plaintext into "tokens", which are natural numbers. All the unique tokens found in a corpus are listed in a token vocabulary. The token vocabulary can also include some other special tokens, relative to the use case. The size of the token vocabulary, in the case of GPT-3.5 and GPT-4, is 100258 (100000 from BPE algorithm and 258 included as special tokens).


Suppose we are encoding the previous example of "aaabdaaabac", with a specified vocabulary size of 6, then it would first be encoded as "0, 0, 0, 1, 2, 0, 0, 0, 1, 0, 3" with a vocabulary of "a=0, b=1, d=2, c=3". Then it would proceed as before, and obtain "4, 5, 2, 4, 5, 0, 3" with a vocabulary of "a=0, b=1, d=2, c=3, aa=4, ab=5".

So far this is essentially the same as before. However, if we only had specified a vocabulary size of 5, then the process would stop at vocabulary "a=0, b=1, d=2, c=3, aa=4", so that the example would be encoded as "4, 0, 1, 2, 4, 0, 1, 0, 3". Conversely, if we had specified a vocabulary size of 8, then it would be encoded as "7, 6, 0, 3", with a vocabulary of "a=0, b=1, d=2, c=3, aa=4, ab=5, aaab=6, aaabd=7". 

This is not maximally compressed, because modified BPE does not aim for maximum compression. 

Instead, it aims for an encoding that is efficient and practical for language model training.




**Output of Byte Pair Encoding (BPE)** = `A Vocabulary!`

Vocabulary size is one of the most critical "architectural" hyperparameters you choose before you ever hit the "Train" button.

It is a classic Goldilocks problem: too small and the model is "illiterate" (slow and wordy); too large and the model is "bloated" (heavy and memory-hungry).

### The Trade-offs of Vocab Size

When a researcher is designing a new LLM like Llama 3 or GPT-4, they have to decide on this number (V) based on several factors:
Why you’d want a LARGE vocab (V≈100k−250k):

* Information Density: One token can represent a whole word like Anthropology. This saves space in the context window (the model’s "short-term memory").
* Inference Speed: Since the model generates text token-by-token, a larger vocab means it can "say" more per step. Generating "apple" in 1 step is faster than in 3 steps (ap + pl + e).
* Multilingualism: You can "reserve" blocks of the vocabulary for different scripts (Cyrillic, Kanji, Arabic) so they don't get broken down into ugly, inefficient byte-level fragments.

Why you’d want a SMALL vocab (V≈32k−50k):
* **Memory Efficiency:** 
    - The "Embedding Layer" is essentially a giant lookup table of shape (V,d) where d is the model's hidden dimension.
* **The "Long Tail" Problem:** 
    - If your vocab is too big, many tokens (like very rare medical terms) might only appear a few times in the training data. The model won't see them enough to learn a good mathematical representation for them.

### The Hidden Cost: The "Softmax" Bottleneck

There is a technical reason why we don't just make the vocab 1 million tokens.

At every single step of text generation, the model calculates a probability distribution across the entire vocabulary to decide which word comes next. 

This involves a Softmax operation:

$$P(y_i) = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}}$$

If V (the vocab size) is `256,000`, the model has to do a massive amount of math just to pick one word. 

This makes the final layer of the neural network incredibly "expensive" in terms of raw compute.

### How it's chosen in practice

Usually, the team training the model will:
* Run the BPE algorithm on a representative sample of their training data (e.g., 100GB of internet text).
* Monitor the "Compression Ratio": How many bytes of text, on average, does one token represent?
* Find the "Elbow": They look for the point where increasing the vocab size further doesn't significantly improve compression but does significantly increase the model's memory footprint.

| Model | Vocab Size | Approx. Bytes per Token |
| :--- | :--- | :--- |
| GPT-2 | 50,257 | ~3.5 |
| Llama 3 | 128,256 | ~4.2 |
| Gemma 2 | 256,000 | ~4.8+ |


### The "Special" Tokens

When setting this hyperparameter, researchers also "hard-code" special tokens that aren't part of the BPE merge process. These are usually added at the very end of the vocab:
* `<|endoftext|>`: Tells the model to stop talking.
* `<|image|>`: Used in multimodal models to signal a picture is coming.
* `<|padding|>`: Used to make sentences the same length during training.

## Implementation

In [44]:
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."

# raw bytes that represeent this string in utf-8 encoding
# converts to list of integers in range (0, 255) for convenience
tokens = list(map(int, text.encode("utf-8")))

In [45]:
print(f"Text Length: {len(text)}")
print(f"Token Length: {len(tokens)}")

Text Length: 533
Token Length: 616


Note: the length of tokens > length of text because the unicode emoji is more than 1 byte (compared to alphanumeric characters)

In [46]:
print(tokens)

[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140, 240, 159, 135, 169, 226, 128, 140, 240, 159, 135, 170, 33, 32, 240, 159, 152, 132, 32, 84, 104, 101, 32, 118, 101, 114, 121, 32, 110, 97, 109, 101, 32, 115, 116, 114, 105, 107, 101, 115, 32, 102, 101, 97, 114, 32, 97, 110, 100, 32, 97, 119, 101, 32, 105, 110, 116, 111, 32, 116, 104, 101, 32, 104, 101, 97, 114, 116, 115, 32, 111, 102, 32, 112, 114, 111, 103, 114, 97, 109, 109, 101, 114, 115, 32, 119, 111, 114, 108, 100, 119, 105, 100, 101, 46, 32, 87, 101, 32, 97, 108, 108, 32, 107, 110, 111, 119, 32, 119, 101, 32, 111, 117, 103, 104, 116, 32, 116

In [47]:
from typing import Dict, Tuple, List, Optional


def get_stats(idxs: List[int]) -> Dict[Tuple[int, int], int]:
    """_summary_

    :param idxs: list of integers representing token index
    :return: _description_
    """
    counts = {}
    # O(n^2) for every pair of token to other tokens other than itself
    for pair in zip(idxs, idxs[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


pairs_dict = get_stats(tokens)
pairs_dict

{(239, 188): 1,
 (188, 181): 1,
 (181, 239): 1,
 (239, 189): 6,
 (189, 142): 1,
 (142, 239): 1,
 (189, 137): 1,
 (137, 239): 1,
 (189, 131): 1,
 (131, 239): 1,
 (189, 143): 1,
 (143, 239): 1,
 (189, 132): 1,
 (132, 239): 1,
 (189, 133): 1,
 (133, 33): 1,
 (33, 32): 2,
 (32, 240): 3,
 (240, 159): 15,
 (159, 133): 7,
 (133, 164): 1,
 (164, 240): 1,
 (133, 157): 1,
 (157, 240): 1,
 (133, 152): 1,
 (152, 240): 1,
 (133, 146): 1,
 (146, 240): 1,
 (133, 158): 1,
 (158, 240): 1,
 (133, 147): 1,
 (147, 240): 1,
 (133, 148): 1,
 (148, 226): 1,
 (226, 128): 12,
 (128, 189): 1,
 (189, 32): 1,
 (159, 135): 7,
 (135, 186): 1,
 (186, 226): 1,
 (128, 140): 6,
 (140, 240): 6,
 (135, 179): 1,
 (179, 226): 1,
 (135, 174): 1,
 (174, 226): 1,
 (135, 168): 1,
 (168, 226): 1,
 (135, 180): 1,
 (180, 226): 1,
 (135, 169): 1,
 (169, 226): 1,
 (135, 170): 1,
 (170, 33): 1,
 (159, 152): 1,
 (152, 132): 1,
 (132, 32): 1,
 (32, 84): 1,
 (84, 104): 1,
 (104, 101): 6,
 (101, 32): 20,
 (32, 118): 1,
 (118, 101): 3,
 

In [48]:
# let's sort by most common
sorted([(v, k) for k, v in pairs_dict.items()], reverse=True)

[(20, (101, 32)),
 (15, (240, 159)),
 (12, (226, 128)),
 (12, (105, 110)),
 (10, (115, 32)),
 (10, (97, 110)),
 (10, (32, 97)),
 (9, (32, 116)),
 (8, (116, 104)),
 (7, (159, 135)),
 (7, (159, 133)),
 (7, (97, 114)),
 (6, (239, 189)),
 (6, (140, 240)),
 (6, (128, 140)),
 (6, (116, 32)),
 (6, (114, 32)),
 (6, (111, 114)),
 (6, (110, 103)),
 (6, (110, 100)),
 (6, (109, 101)),
 (6, (104, 101)),
 (6, (101, 114)),
 (6, (32, 105)),
 (5, (117, 115)),
 (5, (115, 116)),
 (5, (110, 32)),
 (5, (100, 101)),
 (5, (44, 32)),
 (5, (32, 115)),
 (4, (116, 105)),
 (4, (116, 101)),
 (4, (115, 44)),
 (4, (114, 105)),
 (4, (111, 117)),
 (4, (111, 100)),
 (4, (110, 116)),
 (4, (110, 105)),
 (4, (105, 99)),
 (4, (104, 97)),
 (4, (103, 32)),
 (4, (101, 97)),
 (4, (100, 32)),
 (4, (99, 111)),
 (4, (97, 109)),
 (4, (85, 110)),
 (4, (32, 119)),
 (4, (32, 111)),
 (4, (32, 102)),
 (4, (32, 85)),
 (3, (118, 101)),
 (3, (116, 115)),
 (3, (116, 114)),
 (3, (116, 111)),
 (3, (114, 116)),
 (3, (114, 115)),
 (3, (114, 10

In [49]:
# most common pair is (101, 32)
# what are these tokens?
print(f"'{chr(101)}', '{chr(32)}'")

'e', ' '


### Merging the most common pair as a new token

In [50]:
top_pair = max(pairs_dict, key=pairs_dict.get)
top_pair

(101, 32)

In [51]:
def merge(idxs: List[int], target_pair: Tuple[int, int], new_idx: int) -> List[int]:
    """In the list of ints (idxs), replace all consecutive occurences of token pair with the new token index

    :param idxs: _description_
    :param target_pair: _description_
    :param new_idx: _description_
    :return: _description_
    """

    new_idxs = []
    i = 0

    while i < len(idxs):

        # last token
        if i > len(idxs) - 2:
            print(f"last!: {idxs[i]}")
            new_idxs.append(idxs[i])
            i += 1
            continue

        # get pair
        pair = idxs[i], idxs[i + 1]

        # deterimine if need to merge or not
        if pair == target_pair:
            new_idxs.append(new_idx)
            # we're removing this pair entirely so skip to next untouched pair
            i += 2
        else:
            new_idxs.append(pair[0])
            # increment to next
            i += 1

    return new_idxs


# test this out
ex_li = [5, 6, 6, 7, 9, 1]
target_pair = (6, 7)
merge(ex_li, target_pair, 99)

last!: 1


[5, 6, 99, 9, 1]

In [52]:
tokens_2 = merge(tokens, top_pair, 256)
print(f"Tokens_2 Length: {len(tokens_2)}")
print(tokens_2)

last!: 46
Tokens_2 Length: 596
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140, 240, 159, 135, 169, 226, 128, 140, 240, 159, 135, 170, 33, 32, 240, 159, 152, 132, 32, 84, 104, 256, 118, 101, 114, 121, 32, 110, 97, 109, 256, 115, 116, 114, 105, 107, 101, 115, 32, 102, 101, 97, 114, 32, 97, 110, 100, 32, 97, 119, 256, 105, 110, 116, 111, 32, 116, 104, 256, 104, 101, 97, 114, 116, 115, 32, 111, 102, 32, 112, 114, 111, 103, 114, 97, 109, 109, 101, 114, 115, 32, 119, 111, 114, 108, 100, 119, 105, 100, 101, 46, 32, 87, 256, 97, 108, 108, 32, 107, 110, 111, 119, 32, 119, 256, 111, 117, 103, 104, 116, 

## Putting it All Together

In [ ]:
import heapq
import regex as re


class BytePairEncodingTokenizer:
    """Core Principle: the more steps we take, the larger the vocab and shorter sequence length.
    * Hyperparameter to find the sweet spot
    * e.g., GPT-4 uses about 100k tokens
    * Increase in vocab -> increased compression ratio -> shorter sequence length
    """

    def __init__(
        self,
        target_vocab_size: int,
        num_byte_values: int = 256,
        special_tokens: Optional[Dict[str, int]] = None,
    ) -> int:
        """_summary_

        :param target_vocab_size: _description_
        :param num_byte_values: _description_, defaults to 256
        :param special_tokens:
        :return: _description_
        """
        self.target_vocab_size = target_vocab_size
        self.num_byte_values = num_byte_values
        self.num_merges = self.target_vocab_size - self.num_byte_values
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

        # notice how we use bytes([i]) because if you just do bytes(i),
        # it no longer retains i as its own element so bytes(1)+ bytes(2) = bytes(3)
        # instead of bytes([1]) + bytes([2]) = bytes([1, 2]) <-- what we want
        self.vocab = {i: bytes([i]) for i in range(self.num_byte_values)}

        # store special tokens
        self.special_tokens = special_tokens or {}
        self.inverse_special_tokens = {v: k for k, v in self.special_tokens.items()}

        # build special token regex
        # will look for special tokens explicitly
        self.special_pat = None
        if self.special_tokens:
            special_pattern = "|".join(re.escape(k) for k in self.special_tokens.keys())
            self.special_pat = re.compile(f"({special_pattern})")

        # to be calculated
        self.merges = {}
        self.n_tokens_old = 0
        self.n_tokens_new = 0
        self.compression_ratio = 1.0

    def get_stats(self, idxs: List[int]) -> Dict[Tuple[int, int], int]:
        """_summary_

        :param idxs: list of integers representing token index
        :return: _description_
        """
        counts = {}
        # O(n) for every pair of token to other tokens other than itself
        # because zip creates a sliding window of adjacent pairs only
        for pair in zip(idxs, idxs[1:]):
            counts[pair] = counts.get(pair, 0) + 1
        return counts

    def merge(
        self,
        idxs: List[int],
        target_pair: Tuple[int, int],
        new_idx: int,
    ) -> List[int]:
        """In the list of ints (idxs), replace all consecutive occurences of token pair with the new token index

        :param idxs: _description_
        :param target_pair: _description_
        :param new_idx: _description_
        :return: _description_
        """

        new_idxs = []
        i = 0

        while i < len(idxs):
            # last token
            if i > len(idxs) - 2:
                new_idxs.append(idxs[i])
                i += 1
                continue

            # get pair
            pair = idxs[i], idxs[i + 1]

            # deterimine if need to merge or not
            if pair == target_pair:
                new_idxs.append(new_idx)
                # we're removing this pair entirely so skip to next untouched pair
                i += 2
            else:
                new_idxs.append(pair[0])
                # increment to next
                i += 1

        return new_idxs

    def train(self, corpus: str) -> List[int]:
        """Scans a massive string to find patterns and builds the self.merges dictionary.
        preserving special tokens without allowing them to be merged.

        :param corpus: _description_
        :return: _description_
        """
        # 1. Split by special tokens first to protect them
        if self.special_pat:
            parts = self.special_pat.split(corpus)
        else:
            parts = [corpus]

        # 2. Prepare token chunks
        token_chunks = []
        for part in parts:
            if part in self.special_tokens:
                # Wrap special token ID in a list so it's a standalone "chunk"
                token_chunks.append([self.special_tokens[part]])
            elif part:
                # Normal text: use self.pat and encode to bytes
                sub_chunks = self.pat.findall(part)
                for sc in sub_chunks:
                    token_chunks.append(list(sc.encode("utf-8")))

        self.n_tokens_old = sum(len(chunk) for chunk in token_chunks)
        self.merges = {}

        # 3. Main Training Loop
        for i in range(self.num_merges):
            stats = {}
            for chunk in token_chunks:
                # If a chunk is a single special token, it has no pairs to count
                if len(chunk) < 2:
                    continue

                # Standard BPE pair counting
                for pair in zip(chunk, chunk[1:]):
                    stats[pair] = stats.get(pair, 0) + 1

            if not stats:
                print("No more pairs to merge. Stopping early.")
                break

            # 4. Find the Global Winner
            top_pair = max(stats, key=stats.get)
            new_token_idx = self.num_byte_values + i

            # 5. Record the rule
            self.merges[top_pair] = new_token_idx
            self.vocab[new_token_idx] = self.vocab[top_pair[0]] + self.vocab[top_pair[1]]

            # 6. Apply merge to all chunks
            # Note: special token chunks (length 1) are untouched by this
            token_chunks = [self.merge(chunk, top_pair, new_token_idx) for chunk in token_chunks]

            # Preserving your print statement!
            print(f"Merge {i+1}/{self.num_merges}: {top_pair} -> {new_token_idx} (Freq: {stats[top_pair]})")

        # 7. Final Flattening (This now includes your special tokens!)
        final_tokens = [t for chunk in token_chunks for t in chunk]

        self.n_tokens_new = len(final_tokens)
        self.compression_ratio = round(self.n_tokens_old / self.n_tokens_new, 2)

        return final_tokens

    def decode(self, token_idxs: List[int]) -> str:
        """_summary_

        :param token_idxs: _description_
        :return: _description_
        """
        part_bytes = []
        for idx in token_idxs:
            if idx in self.inverse_special_tokens:
                # Handle special tokens
                part_bytes.append(self.inverse_special_tokens[idx].encode("utf-8"))
            elif idx in self.vocab:
                # Handle BPE/byte tokens
                part_bytes.append(self.vocab[idx])
            else:
                # Fallback for unknown IDs
                continue

        tokens = b"".join(part_bytes)
        text = tokens.decode("utf-8", errors="replace")

        return text

    def _encode_normal_text(self, text: str) -> List[int]:
        ## PRO WAY (O(N * logN)) -- PER text chunk
        # greedy algorithm that uses priority queue

        # 1. Use regex to split the text into meaningful chunks
        # This prevents "leaking" merges across word/punctuation boundaries
        text_chunks = re.findall(self.pat, text)
        all_tokens = []

        for chunk in text_chunks:
            chunk_bytes = list(chunk.encode("utf-8"))

            # Simple case: 0 or 1 byte chunks can't be merged
            if len(chunk_bytes) < 2:
                all_tokens.extend(chunk_bytes)
                continue

            # 1. Initialize Linked List for THIS chunk
            # We use a doubly linked list approach (via indices) to make
            # deletions and insertions O(1)
            data = [[i - 1, byte, i + 1] for i, byte in enumerate(chunk_bytes)]
            data[0][0] = None
            data[-1][2] = None

            pq = []

            def add_pair(i, j):
                if i is not None and j is not None:
                    pair = (data[i][1], data[j][1])
                    if pair in self.merges:
                        # We use the merge index (e.g. 256, 257) as the priority
                        heapq.heappush(pq, (self.merges[pair], i, j))

            # 2. Initial heap population
            for i in range(len(data) - 1):
                add_pair(i, i + 1)

            # 3. Process Merges for this chunk
            while pq:
                rank, i, j = heapq.heappop(pq)
                if data[i][2] != j or data[j][0] != i:
                    continue

                # Perform the merge: update the left node with the new token
                data[i][1] = rank

                # Update the linked list pointers to "remove" node j
                next_of_j = data[j][2]
                data[i][2] = next_of_j
                if next_of_j is not None:
                    data[next_of_j][0] = i

                # 4. Update Neighbors
                # New pairs might have been formed with the nodes to the left and right
                # Pair with the node before the merge
                add_pair(data[i][0], i)
                # Pair with the node after the merge
                add_pair(i, next_of_j)

            # 5. Reconstruct this chunk's tokens
            curr = 0
            while data[curr][0] is not None:
                curr = data[curr][0]

            while curr is not None:
                # Append to our GLOBAL list
                all_tokens.append(data[curr][1])
                curr = data[curr][2]

        return all_tokens

    def encode(self, text: str) -> List[int]:
        """Method to take an input string of text and return a list of tokenized indexes.

        IMPORTANT: order matters. earlier merges must happen before later merges

        The "Pro" Way (tiktoken)
        ------------------------
        * use a Min-Heap (Priority Queue).
        * put every possible merge found in the text into a heap, sorted by their "rank" (when they were learned).
        * pop the best one, perform the merge.
        * only update the heap for the new pairs created by that specific merge.
        * this makes the encoding process extremely fast—almost O(NlogN)—allowing it to tokenize millions of words per second.

        :param text: _description_
        :return: _description_
        """

        # no special tokens
        if not self.special_pat:
            return self._encode_normal_text(text)

        # split text into special tokens and normal text
        parts = self.special_pat.split(text)
        final_tokens = []

        for part in parts:
            if part in self.special_tokens:
                # Part is a special token: use its ID directly
                final_tokens.append(self.special_tokens[part])
            elif part:
                # Part is normal text: process with BPE
                final_tokens.extend(self._encode_normal_text(part))

        return final_tokens

In [64]:
bpe = BytePairEncodingTokenizer(target_vocab_size=276)

tokens_3 = bpe.train(text)
print(f"\nTokens_3 Length: {len(tokens_3)}")
print(tokens_3)

Merge 1/20: (240, 159) -> 256 (Freq: 15)
Merge 2/20: (226, 128) -> 257 (Freq: 12)
Merge 3/20: (105, 110) -> 258 (Freq: 12)
Merge 4/20: (32, 97) -> 259 (Freq: 10)
Merge 5/20: (32, 116) -> 260 (Freq: 9)
Merge 6/20: (260, 104) -> 261 (Freq: 8)
Merge 7/20: (256, 133) -> 262 (Freq: 7)
Merge 8/20: (256, 135) -> 263 (Freq: 7)
Merge 9/20: (97, 114) -> 264 (Freq: 7)
Merge 10/20: (239, 189) -> 265 (Freq: 6)
Merge 11/20: (257, 140) -> 266 (Freq: 6)
Merge 12/20: (266, 263) -> 267 (Freq: 6)
Merge 13/20: (101, 114) -> 268 (Freq: 6)
Merge 14/20: (111, 114) -> 269 (Freq: 6)
Merge 15/20: (97, 110) -> 270 (Freq: 6)
Merge 16/20: (258, 103) -> 271 (Freq: 6)
Merge 17/20: (32, 115) -> 272 (Freq: 5)
Merge 18/20: (32, 258) -> 273 (Freq: 5)
Merge 19/20: (100, 101) -> 274 (Freq: 5)
Merge 20/20: (117, 115) -> 275 (Freq: 5)

Tokens_3 Length: 467
[239, 188, 181, 265, 142, 265, 137, 265, 131, 265, 143, 265, 132, 265, 133, 33, 32, 262, 164, 262, 157, 262, 152, 262, 146, 262, 158, 262, 147, 262, 148, 257, 189, 32, 26

### Observation

After 20 steps (merges), we increased our vocabulary size to 276 from 256 and then decreased our sequence length from `616 tokens` to `451 tokens`.

In [65]:
bpe.merges

{(240, 159): 256,
 (226, 128): 257,
 (105, 110): 258,
 (32, 97): 259,
 (32, 116): 260,
 (260, 104): 261,
 (256, 133): 262,
 (256, 135): 263,
 (97, 114): 264,
 (239, 189): 265,
 (257, 140): 266,
 (266, 263): 267,
 (101, 114): 268,
 (111, 114): 269,
 (97, 110): 270,
 (258, 103): 271,
 (32, 115): 272,
 (32, 258): 273,
 (100, 101): 274,
 (117, 115): 275}

NOTE: notice how we've even performed merges on newly created token indexes

In [66]:
bpe.compression_ratio

1.32

Note, the Tokenizer is a completely separate, independent module from the LLM. 
* It has its own training dataset of text (which could be different from that of the LLM), on which you train the vocabulary using the Byte Pair Encoding (BPE) algorithm. 
* It then translates back and forth between raw text and sequences of tokens. The LLM later only ever sees the tokens and never directly deals with any text.

In [67]:
bpe.vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [68]:
bpe.decode([271])

'ing'

In [69]:
bpe.encode("ing")

[271]

In [70]:
print(bpe.encode("The very name strikes fear and awe"))
bpe.decode(
    [
        84,
        104,
        101,
        32,
        118,
        268,
        121,
        32,
        110,
        97,
        109,
        101,
        272,
        116,
        114,
        105,
        107,
        101,
        115,
        32,
        102,
        101,
        264,
        259,
        110,
        100,
        259,
        119,
        101,
    ]
)

[84, 104, 101, 32, 118, 268, 121, 32, 110, 97, 109, 101, 272, 116, 114, 105, 107, 101, 115, 32, 102, 101, 264, 259, 110, 100, 259, 119, 101]


'The very name strikes fear and awe'

## Handling punctuation

In [71]:
import regex as re

gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2pat, "Hello've world123 how's are you!!!?"))

['Hello', "'ve", ' world', '123', ' how', "'s", ' are', ' you', '!!!?']


## Special Tokens

In [73]:
special_tokens = {
    "<|endoftext|>": 50000,
    "<|pad|>": 50001,
    "<|system|>": 50002,
    "<|user|>": 50003,
    "<|assistant|>": 50004,
}

bpe = BytePairEncodingTokenizer(
    target_vocab_size=276,
    special_tokens=special_tokens,
)

text = "<|system|>You are a helpful assisant.<|user|>Hello there, how are you? 😀<|endoftext|>"

tokens = bpe.train(text)
print(f"\ntokens Length: {len(tokens)}")
print(tokens)

encoded_tokens = bpe.encode(text)
reconstructed_string = bpe.decode(encoded_tokens)
reconstructed_string

Merge 1/20: (32, 97) -> 256 (Freq: 4)
Merge 2/20: (114, 101) -> 257 (Freq: 3)
Merge 3/20: (111, 117) -> 258 (Freq: 2)
Merge 4/20: (256, 257) -> 259 (Freq: 2)
Merge 5/20: (32, 104) -> 260 (Freq: 2)
Merge 6/20: (101, 108) -> 261 (Freq: 2)
Merge 7/20: (89, 258) -> 262 (Freq: 1)
Merge 8/20: (260, 261) -> 263 (Freq: 1)
Merge 9/20: (263, 112) -> 264 (Freq: 1)
Merge 10/20: (264, 102) -> 265 (Freq: 1)
Merge 11/20: (265, 117) -> 266 (Freq: 1)
Merge 12/20: (266, 108) -> 267 (Freq: 1)
Merge 13/20: (256, 115) -> 268 (Freq: 1)
Merge 14/20: (268, 115) -> 269 (Freq: 1)
Merge 15/20: (269, 105) -> 270 (Freq: 1)
Merge 16/20: (270, 115) -> 271 (Freq: 1)
Merge 17/20: (271, 97) -> 272 (Freq: 1)
Merge 18/20: (272, 110) -> 273 (Freq: 1)
Merge 19/20: (273, 116) -> 274 (Freq: 1)
Merge 20/20: (72, 261) -> 275 (Freq: 1)

tokens Length: 31
[50002, 262, 259, 256, 267, 274, 46, 50003, 275, 108, 111, 32, 116, 104, 101, 257, 44, 260, 111, 119, 259, 32, 121, 258, 63, 32, 240, 159, 152, 128, 50000]


'<|system|>You are a helpful assisant.<|user|>Hello there, how are you? 😀<|endoftext|>'